# 第4章章节实践：ACL 调用流程

## 本节学习目标

本实践要求综合运用本章知识完成可复现的工程任务。请保留命令、参数、正确性结果和分析结论。

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 必要背景与实验材料

本实践基于本章 `src/` 中的课程工程副本。开始前应完成前面各小节，并能解释工程的关键源码、构建入口、正确性门槛和计时字段。SpMV 还必须按 `EXPERIMENT_GUIDE.md` 准备与当前 CANN 配套的 `ops-sparse`；缺少可选组件不是 CPU/stub 结果的替代理由。

## 实践任务

1. 从源码恢复 GEMM 的完整 ACLNN 调用顺序
2. 从源码恢复 SpMV 的 handle/descriptor 调用顺序
3. 选择一个参数变化运行正确性实验
4. 解释两条 API 不能机械互换的原因

## 核心知识与关键源码解析

本章实践只调整 Notebook 中的运行参数和实验配置，不修改 `src/`。请保持输入与正确性阈值一致，每次只改变一个变量，并说明它如何影响数据流和性能。

## 实验记录模板

执行下面的准备命令，然后在目标环境中完成任务。不要把参考答案中的结论当作实测结果。

In [ ]:
%%bash
set -e
cd src/acl_operator_calls
bash GEMM-acl/scripts/build.sh
grep -q 'ACL_C_STUB:BOOL=OFF' GEMM-acl/build/CMakeCache.txt
bash GEMM-acl/scripts/run.sh --m 1024 --k 1024 --n 1024 --warmup 3 --repeat 10
bash SpMV-acl/scripts/build.sh
grep -q 'ACL_C_STUB:BOOL=OFF' SpMV-acl/build/CMakeCache.txt
bash SpMV-acl/scripts/run.sh --rows 100000 --cols 100000 --nnz 1000000 --warmup 3 --repeat 10

## 评价标准

必须使用工程实际 API 名称，结果需包含退出码和误差；stub 输出不得作为 ACL 计算结果。

## 查看参考答案

参考答案给出方法和判断依据，不提供虚构的固定性能数字。

## 预期现象与结果分析

正确性门槛应首先通过；性能结果随硬件、软件栈和系统负载变化。若修改后没有加速或出现退化，也应依据阶段计时、通信次数或资源竞争给出解释。

## 实践小结

完成报告时，应明确实验环境、唯一修改变量、正确性门槛、计时口径和观察到的限制。

## 工程实践提交物与完成标准

章测必须基于 `src/acl_operator_calls/`，不得只回答概念题。操作链：识别 real/stub → 追踪 GEMM 的初始化、Tensor、Workspace/Executor、执行、同步、验证、释放 → 追踪 SpMV 的 handle、CSR descriptor、执行、验证、释放 → 比较 API。

提交物：实际命令与环境；阅读或修改的真实文件/函数/参数；字段为“实验、Backend、规模、Workspace、时间、相对误差、释放检查”的结果表；正确性判据；基于数据的结论。性能数字不作为固定答案。

完成标准：命令指向真实脚本或可执行文件，数据来自同口径运行，并能解释结果。


## 四类考核

以下四题中，客观题答案唯一，凭 `src/acl_operator_calls/` 源码即可判定；简单/中等/困难题基于本章实验，要求用命令、输出或计算过程作为证据，不接受无证据的概念回答。

### 1. 客观题

（1）单选：`SpMV-acl/src/main.cpp` 中执行 SpMV 的实际调用是（　）

A. 先查询 workspace 大小并用 `aclrtMalloc` 分配，再调用 `aclsparseSpMV`

B. 直接调用 `aclsparseSpMV(handle, ACL_SPARSE_OP_NON_TRANSPOSE, &alpha, matrix, vec_x, &beta, vec_y, ACL_FLOAT, ACL_SPARSE_SPMV_ALG_DEFAULT, nullptr)`，不查询、不分配 workspace

C. 先用额外 API 手动写入 CSR 数据，再调用 `aclsparseSpMV`

D. 把输出向量当作稠密矩阵，用稠密矩阵乘 API 计算

（2）判断（对/错）：SpMV 的释放顺序与创建顺序相反：`aclsparseDestroyDnVec`（vec_x、vec_y）→ `aclsparseDestroySpMat`（matrix）→ `aclsparseDestroy`（handle）。（　）

### 2. 简单题

给出一次成功运行的完整证据：构建日志中 `ACL_C_STUB=OFF`（或 real backend）的证明、运行命令、退出码、相对误差与 `1e-6` 门槛的比较。

### 3. 中等题

用源码中 `copy_to_device`（H2D）与 `copy_to_host`（D2H）的位置，以及运行输出的 `Timing scope`、`ACL SpMV time`、`CPU Reference Error`、`Workspace bytes` 字段描述 SpMV 数据流——SpMV 输出没有独立的 H2D/D2H 计时分项，不要虚构。分别在 GEMM 与 SpMV 各自固定输入下运行一次，核验真实调用链、同步、时间字段、误差与 workspace 语义（GEMM 有 workspace/executor；SpMV 直接 `aclsparseSpMV(..., nullptr)`）。GEMM 与 SpMV 是不同数学运算、不同输入表示，不能把两者耗时当作可比性能结论；对比的是 API/资源生命周期与证据字段；结论必须引用实测输出字段。

### 4. 困难题

构造一个错误场景（如跳过资源初始化，或错误释放顺序/重复释放），记录 ACL 错误码与日志层次，说明错误路径上的清理要求；同时用运行时输出（非变量名）证明本次计算发生在真实 NPU 后端。
